In [1]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

from utils import protein
from utils.geometry import compute_rmsd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)
/data/cb/mihirb14/projects/BoltzDesign1/utils/geometry.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(False)


In [ ]:
motif = "1prw"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
root_dir = f"./out/onemotif_twostates/{motif}/"

rows = []
for design_dir in sorted(glob.glob(os.path.join(root_dir, "design*"))):
    design_name = os.path.basename(design_dir)
    
    with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
        motif_mask = pickle.load(f)["motif_mask"]

    for state in [0, 1]:
        with open(os.path.join(design_dir, f"state{state}.pkl"),"rb") as f:
            outdict = pickle.load(f)
        for sample_idx in range(5): 
            pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
            if not os.path.exists(pdb_file):
                continue
            rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)
            rows.append({
                "design": design_name,
                "state": state,
                "sample": sample_idx,
                "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                "ptm": outdict["ptm"].cpu().numpy()[sample_idx]
                # prolly should save iptm iplddt for state 1
            })

df = pd.DataFrame(rows)
df.head()


,design,state,sample,motifrmsd,plddt,ptm
0,design0,0,0,5.248249,0.629778,0.358699
1,design0,0,1,5.256164,0.632213,0.370536
2,design0,0,2,5.005022,0.647516,0.404510
3,design0,0,3,4.915047,0.608674,0.365064
4,design0,0,4,5.175341,0.637479,0.389172


In [32]:
# design_dir = f"./out/onemotif_twostates/{motif}/design1/"

# with open(os.path.join(design_dir, f"state1.pkl"),"rb") as f:
#     out = pickle.load(f)
# out

In [10]:
# aggregate
agg = df.groupby(["design", "state"]).agg(
    motifRMSD_mean=("motifrmsd", "mean"),
    motifRMSD_std=("motifrmsd", "std"),
    plddt=("plddt", "mean"),
    ptm=("ptm", "mean")
).reset_index()

# pivot wider
pivot = agg.pivot(index="design", columns="state").reset_index()

# flatten multiindex -> "metric_state"
pivot.columns = ["_".join(map(str, col)).rstrip("_") for col in pivot.columns.to_flat_index()]

# clean up names
pivot = pivot.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))
pivot

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,5.119965,8.210560,0.152706,1.819290,0.631132,0.427174,0.377596,0.294430
1,design1,2.771854,3.671788,0.167299,0.465904,0.503981,0.450029,0.320821,0.393131
2,design2,0.661238,8.539115,0.093913,1.496075,0.628700,0.514518,0.451927,0.545563
3,design3,1.384750,4.687734,0.216496,2.182162,0.615123,0.552483,0.459785,0.493525
4,design4,1.675966,1.861277,0.179673,0.046701,0.689835,0.821882,0.539707,0.860435
5,design5,3.066461,5.353533,0.035418,0.089793,0.804286,0.731925,0.770391,0.772369
6,design6,8.542995,9.189621,0.628534,0.506181,0.486772,0.493825,0.271823,0.457749
7,design7,4.100723,4.687543,0.353821,0.085331,0.668639,0.638811,0.503431,0.595596


In [11]:
import plotly.express as px

fig = px.scatter(
    pivot,
    x="motifRMSD_mean_unbound",
    y="motifRMSD_mean_bound",
    error_x="motifRMSD_std_unbound",
    error_y="motifRMSD_std_bound",
    color="plddt_unbound",
    hover_name="design",
    labels={
        "motifRMSD_mean_unbound": "Unbound motif RMSD",
        "motifRMSD_mean_bound": "Bound motif RMSD"
    },
    title="Unbound vs Bound motif RMSD (±1 std)",
    color_continuous_scale="sunsetdark"
)

fig.add_shape(type="line", x0=0, y0=0, x1=5, y1=5,
              line=dict(color="lightgray", dash="dash"))
fig.update_layout(
    xaxis=dict(range=[0,10]),
    yaxis=dict(range=[0,10], scaleanchor="x"),
    width=600, height=600,
    # plot_bgcolor="white",
    # paper_bgcolor="white"   
)
fig.update_traces(
    error_x=dict(color="gray", thickness=1.0),
    error_y=dict(color="gray", thickness=1.0)
)
fig.show()
